# Multilingual Fake News Detection - Training Notebook

This notebook sets up and runs training for XLM-RoBERTa, MuRIL, or Ensemble models on Google Colab, Kaggle, or local environments with GPU acceleration.

## Setup Steps:
1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → GPU (T4 or A100) [Colab only]
2. **Mount Google Drive**: Run the cell below to access your data [Colab only]
3. **Upload your project**: Either via Drive or direct upload
4. **Run training**: Execute cells sequentially

In [ ]:
import os
import shutil
import sys

# 1. Define paths
input_path = '/kaggle/input/datasets/khushalsainiwal/fnd-dataset-v4/fnd'
working_path = '/kaggle/working/'

# 2. Copy all files from input to working directory
if os.path.exists(input_path):
    # Copying everything to /kaggle/working/
    for item in os.listdir(input_path):
        s = os.path.join(input_path, item)
        d = os.path.join(working_path, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print("Files copied to working directory.")
else:
    print("Error: Input path not found. Please double-check the 'Input' sidebar.")

# 3. Setup Python path so imports work
sys.path.append(working_path)
os.chdir(working_path)

# 4. Verify you see train.py and your folders now
# !ls

# 5. Verify your data and checkpoints are present
print("Files in working directory:", os.listdir())
if os.path.exists('models/checkpoints/'):
    print("Checkpoints found:", os.listdir('models/checkpoints/'))


In [ ]:
# import os
# import shutil

# # 1. Define Paths
# # Check the right sidebar to confirm this exact path for 'checkpoint_source'
# project_source = '/kaggle/input/datasets/khushalsainiwal/fnd-v1/fnd'
# checkpoint_source = '/kaggle/input/notebooks/khushalsainiwal/fnd-kaggle-v3/models/checkpoints/ensemble_resume.pt' 
# working_dir = '/kaggle/working/'

# # 2. Copy project files
# for item in os.listdir(project_source):
#     s = os.path.join(project_source, item)
#     d = os.path.join(working_dir, item)
#     if os.path.isdir(s):
#         shutil.copytree(s, d, dirs_exist_ok=True)
#     else:
#         shutil.copy2(s, d)

# # 3. Inject the checkpoint into the expected folder
# target_dir = '/kaggle/working/models/checkpoints/'
# os.makedirs(target_dir, exist_ok=True)
# shutil.copy2(checkpoint_source, os.path.join(target_dir, 'ensemble_resume.pt'))

# # 4. Verify your data and checkpoints are present
# print("Files in working directory:", os.listdir())
# if os.path.exists('models/checkpoints/'):
#     print("Checkpoints found:", os.listdir('models/checkpoints/'))
    
# print("Ready!")

## 1. Check GPU and System Info

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:

print(f"Working directory: {os.getcwd()}")
!ls -la

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate
!pip install -q langdetect scikit-learn
!pip install -q wandb  # Optional: for experiment tracking

# Verify installations
import transformers
print(f"Transformers version: {transformers.__version__}")

## 3. Verify Data Files

In [ ]:
import pandas as pd

# Check processed data exists
for split in ['train', 'val', 'test']:
    path = f'data/processed/{split}.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"{split:5s}: {len(df):,} rows")
        print(f"  Languages: {df['language'].value_counts().to_dict()}")
        print(f"  Labels: {df['label'].value_counts().to_dict()}")
        print()
    else:
        print(f"❌ Missing: {path}")

## 4. Configure Training

Edit the CONFIG below to choose your model type and hyperparameters.

In [ ]:
# Training configuration
CONFIG = {
    # ── Model Selection ──────────────────────────────────────────────
    'model_type': 'ensemble',              # Options: 'xlm-roberta', 'muril', 'ensemble'
    
    # For standalone models (xlm-roberta or muril):
    'model_name': 'ensemble',  # or 'xlm-roberta-base', 'google/muril-base-cased'
    
    # For ensemble (only used if model_type='ensemble'):
    'xlmr_model_name': 'xlm-roberta-base',    # XLM-RoBERTa model name
    'muril_model_name': 'google/muril-base-cased',
    'xlmr_checkpoint': 'fnd/models/checkpoints/xlm-roberta_best.pt',                    # Path to pre-trained XLM-R checkpoint (optional)
    'muril_checkpoint': 'fnd/models/checkpoints/muril_best.pt',                   # Path to pre-trained MuRIL checkpoint (optional)
    'ensemble_method': 'learned',       # Options: 'weighted_avg', 'max', 'learned'
    'ensemble_weights': None,          # Only for weighted_avg [xlmr_weight, muril_weight] [0.4, 0.6]
    
    # ── Training Hyperparameters ────────────────────────────────────
    'batch_size': 16,                        # Reduce if GPU memory is limited
    'encoder_lr': 2e-5,                      # Learning rate for pretrained encoder
    'classifier_lr': 1e-4,                   # Learning rate for classification head
    'ensemble_lr': 1e-3,                     # Learning rate for ensemble FC (only for ensemble)
    'num_epochs': 3,                         # Increase for better results (3-10 typical)
    'warmup_ratio': 0.1,                     # Warmup proportion of total steps
    'max_length': 512,                       # Token sequence length (128-512)
    'dropout': 0.3,                          # Dropout rate
    'freeze_layers': 6,                      # Number of encoder layers to freeze (0 = train all)
    
    # ── Experiment Tracking ─────────────────────────────────────────
    'use_wandb': False,                      # Set True to use Weights & Biases tracking
    # 'wandb_project': 'fake-news-detection',
    # 'wandb_run_name': 'ensemble-v1',
    
    # ── Paths ───────────────────────────────────────────────────────
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'save_dir': 'models/checkpoints',
}

# Print config
import json
print("Training Configuration:")
print(json.dumps({k: str(v) for k, v in CONFIG.items()}, indent=2))

## 5. Run Training

This executes the main training script.

In [ ]:
# Import training code
import sys
sys.path.insert(0, os.getcwd())

from train import (
    FakeNewsTrainer,
    _normalise_model_type,
    _extract_state_dict,
    _safe_load,
    CONFIG as TRAIN_CONFIG,
)
from data.dataset import MultilingualFakeNewsDataset
from models.xlm_roberta_model import XLMRobertaFakeNewsClassifier
from models.muril_model import MuRILFakeNewsClassifier
from models.ensemble_model import EnsembleFakeNewsClassifier

# Override train.py CONFIG with our Colab CONFIG
TRAIN_CONFIG.update(CONFIG)

print("="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Model: {CONFIG['model_type']}")
print(f"Device: {CONFIG['device']}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"Epochs: {CONFIG['num_epochs']}")
print("="*70 + "\n")

In [ ]:
# Load data
print("Loading data...")
train_df = pd.read_csv('data/processed/train.csv')
val_df = pd.read_csv('data/processed/val.csv')

# Determine tokenizer
model_type = _normalise_model_type(CONFIG['model_type'])
TOKENIZER_MAP = {
    'xlm-roberta': 'xlm-roberta-base',
    'muril': 'google/muril-base-cased',
    'ensemble': 'ensemble',  
}
tokenizer_name = TOKENIZER_MAP[model_type]

# Create datasets
print("Creating datasets...")
train_dataset = MultilingualFakeNewsDataset(
    texts=train_df['cleaned_text'].values,
    labels=train_df['label'].values,
    languages=train_df['language'].values,
    tokenizer_name=tokenizer_name,
    max_length=CONFIG['max_length'],
)
val_dataset = MultilingualFakeNewsDataset(
    texts=val_df['cleaned_text'].values,
    labels=val_df['label'].values,
    languages=val_df['language'].values,
    tokenizer_name=tokenizer_name,
    max_length=CONFIG['max_length'],
)

print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Val dataset:   {len(val_dataset):,} samples")

In [ ]:
# Build model
print(f"\nBuilding {model_type} model...")

common_kwargs = dict(
    num_classes=2,
    dropout=CONFIG['dropout'],
    freeze_layers=CONFIG['freeze_layers'],
)

if model_type == 'xlm-roberta':
    model = XLMRobertaFakeNewsClassifier(
        model_name=CONFIG.get('model_name', 'xlm-roberta-base'),
        **common_kwargs,
    )
elif model_type == 'muril':
    model = MuRILFakeNewsClassifier(
        model_name=CONFIG.get('model_name', 'google/muril-base-cased'),
        **common_kwargs,
    )
else:  # ensemble
    xlmr = XLMRobertaFakeNewsClassifier(
        model_name=CONFIG.get('xlmr_model_name', 'xlm-roberta-base'),
        **common_kwargs,
    )
    muril = MuRILFakeNewsClassifier(
        model_name=CONFIG.get('model_name', 'google/muril-base-cased'),
        **common_kwargs,
    )
    
    # Load optional pre-trained sub-model checkpoints
    device = CONFIG['device']
    for sub_model, ckpt_key, label in [
        (xlmr, 'xlmr_checkpoint', 'XLM-R'),
        (muril, 'muril_checkpoint', 'MuRIL'),
    ]:
        ckpt_path = CONFIG.get(ckpt_key, '').strip()
        if ckpt_path and os.path.isfile(ckpt_path):
            print(f"  Loading {label} checkpoint: {ckpt_path}")
            _safe_load(sub_model, _extract_state_dict(
                torch.load(ckpt_path, map_location=device)
            ))
    
    model = EnsembleFakeNewsClassifier(
        xlmr_model=xlmr,
        muril_model=muril,
        num_classes=2,
        ensemble_method=CONFIG.get('ensemble_method', 'weighted_avg'),
        weights=CONFIG.get('ensemble_weights'),
        freeze_base_models=True,  # Use "False" if you want to train the base models while training ensemble.
    )

param_info = model.count_parameters()
print(f"  Trainable: {param_info['total_trainable']:,}")
print(f"  Frozen:    {param_info['total_frozen']:,}")
print(f"  Total:     {param_info['total']:,}")

# # changes:

# Replace this:
# print(f"  Trainable: {param_info['total_trainable']:,}")
# print(f"  Frozen:    {param_info['total_frozen']:,}")

# With this:
# print(f"  Trainable: {param_info['trainable']:,}")
# print(f"  Frozen:    {param_info['frozen']:,}")
# print(f"  Total:     {param_info['total']:,}")

# In your ensemble model, keys are:

# total_trainable
# total_frozen

# But in XLM-R / MuRIL models, keys are:

# trainable
# frozen
# total

Checkpoint in input directory

In [ ]:
# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         if filename.endswith('.pt') or filename.endswith('.ckpt'):
#             print(f"Found checkpoint at: {os.path.join(dirname, filename)}")


checkpoint in working directory

In [ ]:
# import os
# for dirname, _, filenames in os.walk('/kaggle/working'):
#     for filename in filenames:
#         if filename.endswith('.pt') or filename.endswith('.ckpt'):
#             print(f"Found checkpoint at: {os.path.join(dirname, filename)}")

copy specific checkpoint in working directory

In [ ]:
# import shutil
# import os

# # Replace this with the path found in the step above
# old_ckpt = '/kaggle/input/datasets/khushalsainiwal/fake-news-detection/fnd/models/checkpoints/muril_resume-002.pt'
# new_ckpt_dir = '/kaggle/working/models/checkpoints/'

# os.makedirs(new_ckpt_dir, exist_ok=True)
# shutil.copy(old_ckpt, os.path.join(new_ckpt_dir, 'muril_resume.pt'))
# print("Checkpoint copied to writable directory!")


In [ ]:
# Check for resumable checkpoint
resume_checkpoint = None
checkpoint_dir = CONFIG['save_dir']
resume_path = os.path.join(checkpoint_dir, f"{model_type}_resume.pt")

if os.path.exists(resume_path):
    print(f"\n{'='*70}")
    print("RESUMABLE CHECKPOINT FOUND")
    print(f"{'='*70}")
    print(f"Path: {resume_path}")
    
    try:
        resume_checkpoint = torch.load(resume_path, map_location=CONFIG['device'])
        print(f"Last completed epoch: {resume_checkpoint.get('epoch', 0)}")
        print(f"Best val F1 so far: {resume_checkpoint.get('best_val_f1', 0):.4f}")
        print(f"\nDo you want to resume from this checkpoint?")
        print("  YES: Training will continue from epoch {}".format(resume_checkpoint.get('epoch', 0) + 1))
        print("  NO:  Training will start fresh (checkpoint will be backed up)")
        
        # Auto-resume in Colab (since user can't interact)
        # To disable auto-resume, set resume_training = False
        resume_training = True
        
        if not resume_training:
            # Backup old checkpoint
            import shutil
            from datetime import datetime
            backup_name = f"{model_type}_resume_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pt"
            shutil.copy2(resume_path, os.path.join(checkpoint_dir, backup_name))
            print(f"\nOld checkpoint backed up to: {backup_name}")
            resume_checkpoint = None
        else:
            print(f"\n✓ Resuming training from epoch {resume_checkpoint.get('epoch', 0) + 1}")
    except Exception as e:
        print(f"\n⚠ Error loading checkpoint: {e}")
        print("Starting fresh training...")
        resume_checkpoint = None
    print(f"{'='*70}\n")
else:
    print("No resumable checkpoint found. Starting fresh training.\n")

# Create trainer
trainer = FakeNewsTrainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    model_type=model_type,
    device=CONFIG['device'],
    batch_size=CONFIG['batch_size'],
    encoder_lr=CONFIG['encoder_lr'],
    classifier_lr=CONFIG['classifier_lr'],
    ensemble_lr=CONFIG.get('ensemble_lr', 1e-3),
    num_epochs=CONFIG['num_epochs'],
    warmup_ratio=CONFIG['warmup_ratio'],
    use_wandb=CONFIG['use_wandb'],
    config=CONFIG,
    resume_checkpoint=resume_checkpoint,  # NEW: pass checkpoint for resuming
)

print("Trainer initialized. Starting training...\n")

In [ ]:
# Train!
trained_model = trainer.train(save_dir=CONFIG['save_dir'])

print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"Best val F1: {trainer.best_val_f1:.4f} at epoch {trainer.best_epoch}")
print(f"Checkpoint: {CONFIG['save_dir']}/{model_type}_best.pt")

## 6. Quick Evaluation on Test Set

In [ ]:
# Load test data
test_df = pd.read_csv('data/processed/test.csv')

test_dataset = MultilingualFakeNewsDataset(
    texts=test_df['cleaned_text'].values,
    labels=test_df['label'].values,
    languages=test_df['language'].values,
    tokenizer_name=tokenizer_name,
    max_length=CONFIG['max_length'],
)

print(f"Test dataset: {len(test_dataset):,} samples")

In [ ]:
# Evaluate
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import numpy as np

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

trained_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = trainer._unpack_batch(batch)
        logits = trainer._forward(inputs, model_type)
        
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average='weighted', zero_division=0
)

print("\n" + "="*70)
print("TEST SET RESULTS")
print("="*70)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("\n" + classification_report(all_labels, all_preds, target_names=['Fake', 'Real']))
print("="*70)

## 7. Download Trained Model

Download the checkpoint to your local machine.

In [ ]:
import glob
import os

# Look for any file starting with 'muril_best' or 'muril_resume'
search_pattern = f"{CONFIG['save_dir']}/{model_type}_*.pt"
list_of_files = glob.glob(search_pattern)

if list_of_files:
    # Sort files by creation time to get the absolute latest one
    latest_file = max(list_of_files, key=os.path.getctime)
    print(f"Found latest checkpoint: {latest_file}")
    
    # If on Kaggle, use this to download:
    from IPython.display import FileLink
    display(FileLink(latest_file))
    
    # If on Colab, use this:
    # files.download(latest_file)
else:
    print(f"No checkpoints matching {search_pattern} found.")